In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Cell 1: Hardware Setup & Mixed Precision

In [18]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# 1. Enable Mixed Precision for T4 Tensor Cores (Massive Speedup)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# 2. Detect Hardware and activate Dual GPU Strategy
strategy = tf.distribute.MirroredStrategy()
print(f"Number of GPUs Synchronized: {strategy.num_replicas_in_sync}")

# Verify Mixed Precision is active
print(f"Compute dtype: {tf.keras.mixed_precision.global_policy().compute_dtype}")
print(f"Variable dtype: {tf.keras.mixed_precision.global_policy().variable_dtype}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of GPUs Synchronized: 2
Compute dtype: float16
Variable dtype: float32


# Cell 2: Configuration & Dynamic Scaling

In [19]:
# ==========================================
# HYPERPARAMETERS (Maximized VRAM Target)
# ==========================================
IMG_SIZE = 384 
EPOCHS = 15

# Push this from 64 up to 128 (or 192 if it doesn't OOM crash). 
# This will force the VRAM usage to spike towards the 15GB ceiling.
BATCH_SIZE_PER_REPLICA = 128 
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync

DATASET_DIR = "/kaggle/input/competitions/snakeclef2022/"
IMAGE_FOLDER = os.path.join(DATASET_DIR, "SnakeCLEF2022-medium_size/SnakeCLEF2022-medium_size") 
METADATA_CSV = os.path.join(DATASET_DIR, "SnakeCLEF2022-TrainMetadata.csv")

print(f"Global Batch Size set to: {GLOBAL_BATCH_SIZE}")

Global Batch Size set to: 256


# Cell 3: High-Performance Data Pipeline

In [20]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

print("Building tf.data pipeline...")
df = pd.read_csv(METADATA_CSV)

df['file_path'] = df['file_path'].apply(lambda x: os.path.join(IMAGE_FOLDER, x))
df['class_id'] = df['class_id'].astype(int)

# ==========================================
# I COMPLETELY DELETED THE OS.PATH.EXISTS LOOP HERE
# ==========================================

NUM_CLASSES = df['class_id'].nunique()
print(f"Total Unique Classes: {NUM_CLASSES}")

print("Calculating class weights for rare species...")
classes = np.unique(df['class_id'])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=df['class_id'])
class_weight_dict = dict(zip(classes, weights))

def process_data(file_path, label):
    img = tf.io.read_file(file_path)
    # Hardware accelerated decoding
    img = tf.image.decode_jpeg(img, channels=3, dct_method="INTEGER_FAST")
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='bilinear')
    img = tf.cast(img, tf.float32)
    return img, label

# Rebuild distributed dataset
dataset = tf.data.Dataset.from_tensor_slices((df['file_path'].values, df['class_id'].values))
dataset = dataset.shuffle(buffer_size=10000)

# Asynchronous processing
dataset = dataset.map(process_data, num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)

# Dynamic corruption handling (This safely replaces the slow pandas check!)
dataset = dataset.ignore_errors()

dataset = dataset.batch(GLOBAL_BATCH_SIZE, drop_remainder=True)

# Aggressive RAM buffering
dataset = dataset.prefetch(buffer_size=8)

print("Pipeline built successfully with aggressive RAM buffering!")

Building tf.data pipeline...
Total Unique Classes: 1572
Calculating class weights for rare species...
Pipeline built successfully with aggressive RAM buffering!


# Cell 4: Model Architecture (Inside GPU Scope)

In [21]:
print("Cloning architecture to Dual GPUs...")

with strategy.scope():
    input_layer = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="serving_default_input_layer")
    
    backbone = tf.keras.applications.EfficientNetV2S(
        input_tensor=input_layer,
        weights="imagenet",
        include_top=False
    )
    
    backbone.trainable = False
    
    x = layers.GlobalAveragePooling2D()(backbone.output)
    x = layers.Dropout(0.3)(x)
    
    output_layer = layers.Dense(NUM_CLASSES, activation="softmax", dtype=tf.float32, name="species_output")(x)
    
    model = models.Model(inputs=input_layer, outputs=output_layer)
    
    # ==========================================
    # FIX: INCREASED LEARNING RATE
    # ==========================================
    # Scaled up to 2e-3 to match the massive global batch size
    scaled_lr = 2e-3 
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=scaled_lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
        jit_compile=True  # <--- Fuses operations to maximize GPU efficiency
    )

model.summary()

Cloning architecture to Dual GPUs...


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ serving_default_in… │ (None, 384, 384,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 384, 384,  │          0 │ serving_default_… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 192, 192,  │        648 │ rescaling_2[0][0] │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 192, 192,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 192, 192,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 192, 192,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 192, 192,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 192, 192,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 192, 192,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 192, 192,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 192, 192,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 192, 192,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 192, 192,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 192, 192,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 96, 96,    │     20,736 │ block1b_add[0][0] │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 96, 96,    │        384 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 96, 96,    │          0 │ block2a_expand_b

 Total params: 22,345,092 (85.24 MB)

 Trainable params: 2,013,732 (7.68 MB)

 Non-trainable params: 20,331,360 (77.56 MB)

# Cell 5: Callbacks & Training Loop

In [ ]:
checkpoint_path = "/kaggle/working/best_snake_model.keras"
checkpoint = callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor="loss", 
    mode="min",
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

print("Starting Dual-GPU Training...")
history = model.fit(
    dataset,
    epochs=EPOCHS,
    class_weight=class_weight_dict, # <-- Forces AI to learn rare snakes
    callbacks=[checkpoint, reduce_lr]
)

Starting Dual-GPU Training...


2026-06-22 10:00:57.517926: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 72/335


Epoch 1/15
INFO:tensorflow:Collective all_reduce tensors: 2 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
     13/Unknown 41s 1s/step - accuracy: 0.0185 - loss: 9.5592

2026-06-22 10:01:39.971531: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 286/368


     17/Unknown 46s 1s/step - accuracy: 0.0214 - loss: 9.5185

...

1055/1055 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1957 - loss: 3.8393
Epoch 3: loss improved from 4.06479 to 3.38960, saving model to /kaggle/working/best_snake_model.keras

Epoch 3: finished saving model to /kaggle/working/best_snake_model.keras
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1296s 1s/step - accuracy: 0.1934 - loss: 3.3896 - learning_rate: 0.0020
Epoch 4/15
   7/1055 ━━━━━━━━━━━━━━━━━━━━ 21:32 1s/step - accuracy: 0.1916 - loss: 4.3020

2026-06-22 11:06:18.539870: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 286/368


   9/1055 ━━━━━━━━━━━━━━━━━━━━ 21:35 1s/step - accuracy: 0.1928 - loss: 4.3732

2026-06-22 11:06:20.382460: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 72/335


 657/1055 ━━━━━━━━━━━━━━━━━━━━ 8:07 1s/step - accuracy: 0.2192 - loss: 3.5508

# Cell 6: LiteRT (TFLite) Mobile Export & JSON Mapping

In [ ]:
# import json

# print("Converting model to Edge format (Float16 Quantization)...")

# # 1. Export the Model Binary
# converter = tf.lite.TFLiteConverter.from_keras_model(model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# converter.target_spec.supported_types = [tf.float16] 

# tflite_model = converter.convert()
# with open("/kaggle/working/SnakeDetected_efficientnet_model.tflite", "wb") as f:
#     f.write(tflite_model)

# print("Generating Species JSON Mapping from True Schema...")

# # 2. FIX 4: Rebuilt the key extractor to map 'binomial_name' instead of missing parameters
# mapping_df = df[['class_id', 'binomial_name']].drop_duplicates().sort_values('class_id')
# mapping_dict = {}

# for _, row in mapping_df.iterrows():
#     raw_name = str(row['binomial_name']).replace("_", " ").strip()
    
#     # Capitalize the word segments nicely for the UI layout display
#     readable_title = raw_name.title()

#     mapping_dict[str(row['class_id'])] = {
#         "scientific": readable_title,
#         "common": f"{readable_title.split()[0]} Species" # Generates clean fallback (e.g. "Natrix Species")
#     }

# with open("/kaggle/working/species_mapping_updated_final.json", "w") as jf:
#     json.dump(mapping_dict, jf, indent=4)

# print("✅ Complete! Run these blocks to build your custom components.")